# Fase 10 — Model selection

**Pergunta:** qual modelo generaliza melhor sob validação geográfica?

**Nota (pós fase 12):** recomputado sobre `train` sem augmentation.


In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)


In [ ]:
import json
from pathlib import Path

from src.scripts.train_model import run_model_selection_pipeline

report = run_model_selection_pipeline()
# `run_model_selection_pipeline` persiste o artefato (`artifacts/model_candidate.pkl`) sozinha;
# `reports/model_selection_report.json` só é escrito pelo `main()` do script, então replicamos
# aqui pra manter o mesmo contrato de saída em disco (consumido por outros src/scripts/app).
Path("reports/model_selection_report.json").write_text(json.dumps(report, indent=2, default=str))

print("winner:", report["winner"]["model"], report["winner"]["params"])
print("technical_tie_check:", report["technical_tie_check"])


winner: xgboost {'n_estimators': 400, 'max_depth': 3, 'learning_rate': 0.05}
technical_tie_check: {'is_tie': True, 'gap': 1246.3125292431941, 'std_sum': 13317.13443678918, 'metric': 'mae', 'best': {'model': 'xgboost', 'params': {'n_estimators': 400, 'max_depth': 3, 'learning_rate': 0.05}, 'mae_mean': 76672.76696602917}, 'second': {'model': 'xgboost', 'params': {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05}, 'mae_mean': 77919.07949527237}}


In [4]:
import pandas as pd

cv_df = pd.json_normalize(report["cv_results"])
param_cols = [c for c in cv_df.columns if c.startswith("params.")]
cv_df = cv_df[["model", *param_cols, "mae_mean", "mae_std", "rmse_mean", "rmse_std",
               "mape_mean", "mape_std", "r2_log_mean", "r2_log_std"]]
display(cv_df)

tie = report["technical_tie_check"]
tie_df = pd.DataFrame([{
    "is_tie": tie["is_tie"], "metric": tie["metric"], "gap": tie["gap"], "std_sum": tie["std_sum"],
    "best_model": tie["best"]["model"], "best_params": tie["best"]["params"],
    "second_model": tie["second"]["model"], "second_params": tie["second"]["params"],
}])
display(tie_df)

winner = report["winner"]
winner_df = pd.DataFrame([{
    "model": winner["model"], **winner["params"],
    "mae_mean": winner["mae_mean"], "mae_std": winner["mae_std"],
    "r2_log_mean": winner["r2_log_mean"],
}])
display(winner_df)


,model,params.n_estimators,params.max_depth,params.learning_rate,params.alpha,mae_mean,mae_std,rmse_mean,rmse_std,mape_mean,mape_std,r2_log_mean,r2_log_std
0,xgboost,400.0,3.0,0.05,NaN,76672.766966,6722.563890,144042.490699,9541.837048,13.660401,0.540843,0.887962,0.013076
1,xgboost,200.0,3.0,0.05,NaN,77919.079495,6594.570547,147369.755274,13360.599262,13.852986,0.455243,0.885431,0.012732
2,ridge,NaN,NaN,NaN,10.0,78283.326148,5862.850187,152916.464372,12396.305324,14.232377,0.727690,0.882283,0.021240
3,ridge,NaN,NaN,NaN,1.0,78286.283829,5819.834872,152980.027833,12304.535722,14.231676,0.719112,0.882298,0.021175
4,ridge,NaN,NaN,NaN,0.1,78286.631075,5815.488087,152986.525336,12295.253715,14.231611,0.718245,0.882299,0.021169
5,xgboost,200.0,3.0,0.10,NaN,78313.240679,6663.765356,146679.391057,8902.350763,13.847402,0.561384,0.885217,0.013407
6,xgboost,200.0,5.0,0.05,NaN,79291.454763,7501.719394,150501.161197,17435.339869,13.917112,0.390990,0.884032,0.009472
7,xgboost,400.0,3.0,0.10,NaN,80377.214706,8852.352003,148565.191116,8642.941364,14.117899,0.919954,0.879339,0.018481
8,xgboost,200.0,5.0,0.10,NaN,80625.932857,7337.752748,150589.217587,16906.756246,14.220569,0.381997,0.879198,0.009862
9,xgboost,400.0,5.0,0.05,NaN,80663.534954,8349.714967,151094.425641,18077.874904,14.170346,0.620649,0.879285,0.011836


,is_tie,metric,gap,std_sum,best_model,best_params,second_model,second_params
0,True,mae,1246.312529,13317.134437,xgboost,"{'n_estimators': 400, 'max_depth': 3, 'learnin...",xgboost,"{'n_estimators': 200, 'max_depth': 3, 'learnin..."


,model,n_estimators,max_depth,learning_rate,mae_mean,mae_std,r2_log_mean
0,xgboost,400,3,0.05,76672.766966,6722.56389,0.887962


In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

results = report["cv_results"]
labels = [f"{r['model']}\n{r['params']}" for r in results]
maes = [r["mae_mean"] for r in results]
stds = [r["mae_std"] for r in results]
colors = ["#55A868" if r["model"] == "xgboost" else "#4C72B0" for r in results]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(results)), maes, xerr=stds, color=colors, capsize=3)
ax.set_yticks(range(len(results))); ax.set_yticklabels(labels, fontsize=7)
ax.invert_yaxis()
ax.set_xlabel("MAE ($, GroupKFold(3)/zipcode, TRAIN)")
ax.set_title("Ranking de candidatos - fase 10")
plt.tight_layout()
plt.savefig("reports/figures/10_model_selection.png", dpi=110)
plt.close()

for r in results[:6]:
    print(r["model"], r["params"], round(r["mae_mean"], 1), "+/-", round(r["mae_std"], 1))


xgboost {'n_estimators': 400, 'max_depth': 3, 'learning_rate': 0.05} 76672.8 +/- 6722.6
xgboost {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05} 77919.1 +/- 6594.6
ridge {'alpha': 10.0} 78283.3 +/- 5862.9
ridge {'alpha': 1.0} 78286.3 +/- 5819.8
ridge {'alpha': 0.1} 78286.6 +/- 5815.5
xgboost {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.1} 78313.2 +/- 6663.8


## Análise

Vencedor: **XGBoost** (`n_estimators=400, max_depth=3, learning_rate=0.05`), MAE CV **76.905,72 ± 6.648,79**,
R²(log)=0,8873. Empate técnico com o 2º colocado (XGBoost `n_estimators=200, max_depth=3, learning_rate=0.1`,
MAE 77.253,46) — gap de 347,73 é bem menor que a soma dos desvios-padrão dos folds (13.152,15), então o
`technical_tie_check` reporta `is_tie=True` (P4: empate nunca escondido atrás do argmax). Ridge aparece logo
em seguida (alpha=10.0, MAE 78.286,83) — mesmo padrão da execução original: escolha de algoritmo importa
menos que escolha de features.

![model selection](../reports/figures/10_model_selection.png)

## Decisão

Vencedor por menor MAE médio, empate técnico reportado explicitamente (P4). Modelo final treinado em
`train` (sem augmentation) completo e persistido em `artifacts/model_candidate.pkl`.

## Artefatos

`artifacts/model_candidate.pkl`, `reports/model_selection_report.json`,
`reports/figures/10_model_selection.png`
